### Task 1

In [1]:
import pandas as pd

df = pd.read_csv('amazon_products.csv')

In [2]:
df.shape[0]

1426337

In [3]:
df.columns.tolist()

['asin',
 'title',
 'imgUrl',
 'productURL',
 'stars',
 'reviews',
 'price',
 'listPrice',
 'category_id',
 'isBestSeller',
 'boughtInLastMonth']

In [4]:
df.head()

,asin,title,imgUrl,productURL,stars,reviews,price,listPrice,category_id,isBestSeller,boughtInLastMonth
0,B014TMV5YE,"Sion Softside Expandable Roller Luggage, Black...",https://m.media-amazon.com/images/I/815dLQKYIY...,https://www.amazon.com/dp/B014TMV5YE,4.5,0,139.99,0.00,104,False,2000
1,B07GDLCQXV,Luggage Sets Expandable PC+ABS Durable Suitcas...,https://m.media-amazon.com/images/I/81bQlm7vf6...,https://www.amazon.com/dp/B07GDLCQXV,4.5,0,169.99,209.99,104,False,1000
2,B07XSCCZYG,Platinum Elite Softside Expandable Checked Lug...,https://m.media-amazon.com/images/I/71EA35zvJB...,https://www.amazon.com/dp/B07XSCCZYG,4.6,0,365.49,429.99,104,False,300
3,B08MVFKGJM,Freeform Hardside Expandable with Double Spinn...,https://m.media-amazon.com/images/I/91k6NYLQyI...,https://www.amazon.com/dp/B08MVFKGJM,4.6,0,291.59,354.37,104,False,400
4,B01DJLKZBA,Winfield 2 Hardside Expandable Luggage with Sp...,https://m.media-amazon.com/images/I/61NJoaZcP9...,https://www.amazon.com/dp/B01DJLKZBA,4.5,0,174.99,309.99,104,False,400


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1426337 entries, 0 to 1426336
Data columns (total 11 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   asin               1426337 non-null  str    
 1   title              1426336 non-null  str    
 2   imgUrl             1426337 non-null  str    
 3   productURL         1426337 non-null  str    
 4   stars              1426337 non-null  float64
 5   reviews            1426337 non-null  int64  
 6   price              1426337 non-null  float64
 7   listPrice          1426337 non-null  float64
 8   category_id        1426337 non-null  int64  
 9   isBestSeller       1426337 non-null  bool   
 10  boughtInLastMonth  1426337 non-null  int64  
dtypes: bool(1), float64(3), int64(3), str(4)
memory usage: 422.7 MB


In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df.isnull().sum()

asin                 0
title                1
imgUrl               0
productURL           0
stars                0
reviews              0
price                0
listPrice            0
category_id          0
isBestSeller         0
boughtInLastMonth    0
dtype: int64

In [8]:
df.dropna(inplace=True)

In [9]:
df = df.drop(columns=["imgUrl", "productURL", "stars", "reviews", "price", "listPrice", "isBestSeller", "boughtInLastMonth"])

In [10]:
df = df.head(10000)

### Task 2

In [11]:
import re
from nltk.corpus import stopwords
import nltk

nltk.download("stopwords")

stop_words = set(stopwords.words("english"))

def clean_title(text):
    text = text.lower()                          
    text = re.sub(r"[^a-z0-9\s]", " ", text)    
    text = re.sub(r"\s+", " ", text).strip() 

    words = [word for word in text.split() if word not in stop_words]
    text = " ".join(words)

    return text

[nltk_data] Downloading package stopwords to /home/aman/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [12]:
df["clean_text"] = df["title"].apply(clean_title)

In [13]:
df["clean_text"].head()

0    sion softside expandable roller luggage black ...
1    luggage sets expandable pc abs durable suitcas...
2    platinum elite softside expandable checked lug...
3    freeform hardside expandable double spinner wh...
4    winfield 2 hardside expandable luggage spinner...
Name: clean_text, dtype: str

### Task 3

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy import sparse
import joblib

vectorizer = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))

tfidf_matrix = vectorizer.fit_transform(df["clean_text"])

joblib.dump(vectorizer, "tfidf_vectorizer.pkl")
sparse.save_npz("tfidf_matrix.npz", tfidf_matrix)

print(tfidf_matrix.shape)

(10000, 1000)


### Task 4

In [15]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix)

joblib.dump(cosine_sim, "similarity_matrix.pkl")

['similarity_matrix.pkl']

We use cosine similarity because after TF-IDF vectorization, every product title becomes a vector, and cosine similarity measures how similar the directions of two vectors are, regardless of their length.

### Task 5

In [16]:
def recommend(item_name, top_n=5):
    if item_name not in df["title"].values:
        return f"Item '{item_name}' not found in the dataset."

    item_index = df[df["title"] == item_name].index[0]

    # Get the similarity scores for the item
    similarity_scores = list(enumerate(cosine_sim[item_index]))

    sorted_items = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    top_indices = [i[0] for i in sorted_items[1:top_n + 1]]

    return df["title"].iloc[top_indices].tolist()

In [17]:
# Test 1

user_input = "Women's Spinner Mobile Office, Black, One Size"

result = recommend(user_input)
print("Recommended items:")
for item in result:
    print(item)

Recommended items:
Women's Vivian 29" Spinner, Marble Web, One Size
Women's Luggage Adriana 29" Hardside Check in Spinner, Leopard, One Size
Luggage Crimson 29" Hardside Check in Spinner, Black, One Size
Women's Luggage Marie 29" Hardside Check in Spinner, Telescoping Handles, Black Floral Print, One Size
Split Roller 110L - Woodland Floral, One Size


In [18]:
# Test 2

user_input = "Omni 2 Hardside Expandable Luggage with Spinner Wheels, Checked-Large 28-Inch, Arctic Silver"

result = recommend(user_input)
print("Recommended items:")
for item in result:
    print(item)

Recommended items:
Moonlight Hardside Expandable Luggage with Spinner Wheels, Black Marble, Checked-Large 28-Inch
8090 Hardside Expandable Luggage with Spinner Wheels, Black, Checked-Large 28-Inch
Margot Hardside Expandable Luggage with Spinner Wheels, Black, Checked Large 28 Inch
Winfield 2 Hardside Expandable Luggage with Spinner Wheels, Checked-Large 28-Inch, Deep Blue
St. Tropez Hardside Expandable Luggage with Spinner Wheels, Pink, Checked-Large 28 Inch


In [19]:
# Test 3

user_input = "Men's Raid 2.0 Gym Shorts"

result = recommend(user_input)
print("Recommended items:")
for item in result:
    print(item)

Recommended items:
Men's Shorts with Brief Liner, MVP, Gym Shorts for Men, Moisture Wicking Shorts, 5"
5 Pack Mens Shorts, Athletic Gym Shorts Workout Basketball Shorts for Men, SM - 5X
Men's Originals Cotton Pockets, Pull-on Jersey Gym Shorts, 7"
Men's Workout Shorts 7" Running Shorts Athletic Bike Shorts Gym Shorts for Men with Zipper Pocket
Big and Tall Shorts for Men – Side Script Jersey Athletic Gym Shorts
